In [1]:
from init_env import cfg, seed
from dataset import *
import os
import json

In [2]:
# Load data path from config
DATA_PATH = cfg["paths"]["data_path"]

catalog = pd.read_csv(os.path.join(DATA_PATH, "catalog.csv"))
catalog.head()

,dataset,patient_id,original_path,processed_path,laterality,view,pathology
0,cbis_ddsm,cbis_ddsm_P_00047,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.391876...,cbis_ddsm/cbis_ddsm_P_00047_U_U_4d1026b0.png,U,U,malignant
1,cbis_ddsm,cbis_ddsm_P_01682,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.379825...,cbis_ddsm/cbis_ddsm_P_01682_U_U_d148d0ae.png,U,U,malignant
2,cbis_ddsm,cbis_ddsm_P_00119,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.162472...,cbis_ddsm/cbis_ddsm_P_00119_U_U_87fd97ae.png,U,U,benign
3,cbis_ddsm,cbis_ddsm_P_01809,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.390096...,cbis_ddsm/cbis_ddsm_P_01809_U_U_1fec112e.png,U,U,malignant
4,cbis_ddsm,cbis_ddsm_P_00972,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.384230...,cbis_ddsm/cbis_ddsm_P_00972_U_U_dfb5943f.png,U,U,malignant


In [3]:
# Assign numeric labels for pathologies
catalog["label_int"] = catalog["pathology"].map(cfg["labels"])

In [4]:
# Remove unknown-label images
catalog = catalog[catalog["pathology"] != "unknown"]

# Pathology distribution
pathology_counts = catalog['pathology'].value_counts()
print(pathology_counts)

pathology
normal       20841
malignant     6873
benign        4407
Name: count, dtype: int64


In [10]:
catalog = map_ethnicity(catalog)
catalog.groupby('dataset')['ethnicity'].value_counts()

dataset    ethnicity 
cbis_ddsm  us             2857
cmmd       chinese        5202
dmid       us              952
inbreast   portuguese      410
kau_bcmd   saudi          2378
mini_mias  uk              322
vindr      vietnamese    20000
Name: count, dtype: int64

In [11]:
# Collects all pathologies of same patient into a list
patient_pathologies = aggregate_patient_pathologies(catalog)

# Gets patients with more than one pathology
conflicting = patient_pathologies[patient_pathologies["pathologies"].apply(len) > 1]
print("Number of patients with conflicting labels:", len(conflicting))
# Preview
conflicting.head()

Number of patients with conflicting labels: 911


,patient_id,dataset,pathologies
4287,kau_bcmd_2016_BC009024,kau_bcmd,"[normal, malignant]"
4301,kau_bcmd_2016_BC014002,kau_bcmd,"[normal, benign]"
4302,kau_bcmd_2016_BC014181,kau_bcmd,"[normal, malignant]"
4325,kau_bcmd_2016_BC016741,kau_bcmd,"[normal, malignant]"
4341,kau_bcmd_2017_BC0020261,kau_bcmd,"[normal, malignant]"


In [12]:
# Assigns single label based on pathology scheme and separates patients into strata
# Malignant > Benign > Normal
patient_groups = assign_patient_labels(patient_pathologies)
patient_groups.head()

,patient_id,dataset,pathologies,patient_label,strata
0,cbis_ddsm_P_00001,cbis_ddsm,[malignant],malignant,cbis_ddsm_malignant
1,cbis_ddsm_P_00004,cbis_ddsm,[benign],benign,cbis_ddsm_benign
2,cbis_ddsm_P_00005,cbis_ddsm,[malignant],malignant,cbis_ddsm_malignant
3,cbis_ddsm_P_00007,cbis_ddsm,[benign],benign,cbis_ddsm_benign
4,cbis_ddsm_P_00008,cbis_ddsm,[benign],benign,cbis_ddsm_benign


In [13]:
train_df, val_df, test_df = prepare_splits(
    catalog,
    patient_groups,
    train_frac=0.7,
    val_frac=0.15,
    random_state=seed, # For train_test_split
    save_csv=True,
    data_path=DATA_PATH,
    weight_col="ethnicity"
)
train_df.head()

No patient leakage detected!


,dataset,patient_id,original_path,processed_path,laterality,view,pathology,label_int,ethnicity,weights
0,cbis_ddsm,cbis_ddsm_P_00047,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.391876...,../MammoNet32k\processed\cbis_ddsm/cbis_ddsm_P...,U,U,malignant,2.0,us,1.409159
1,cbis_ddsm,cbis_ddsm_P_01682,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.379825...,../MammoNet32k\processed\cbis_ddsm/cbis_ddsm_P...,U,U,malignant,2.0,us,1.409159
2,cbis_ddsm,cbis_ddsm_P_00119,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.162472...,../MammoNet32k\processed\cbis_ddsm/cbis_ddsm_P...,U,U,benign,1.0,us,1.409159
6,cbis_ddsm,cbis_ddsm_P_00419,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.243518...,../MammoNet32k\processed\cbis_ddsm/cbis_ddsm_P...,U,U,benign,1.0,us,1.409159
7,cbis_ddsm,cbis_ddsm_P_00466,cbis_ddsm/jpeg/1.3.6.1.4.1.9590.100.1.2.324735...,../MammoNet32k\processed\cbis_ddsm/cbis_ddsm_P...,U,U,malignant,2.0,us,1.409159


In [14]:
# Extract configuration settings
IMG_SIZE = tuple(cfg["img_size"])
BATCH_SIZE = cfg["batch_size"]
USE_AUTOTUNE = cfg["use_autotune"]

# Shuffle train (Prevent model from learning order of data)
# Augment train (Make model more robust to variations)
train_ds = create_ds_from_df(train_df, shuffle=True, augment=True, batch_size=BATCH_SIZE, use_autotune=USE_AUTOTUNE)
val_ds = create_ds_from_df(val_df, batch_size=BATCH_SIZE, use_autotune=USE_AUTOTUNE)
test_ds = create_ds_from_df(test_df, batch_size=BATCH_SIZE, use_autotune=USE_AUTOTUNE)

In [15]:
class_weights = get_class_weights(train_df["label_int"])

# Dump class weights as json for reuse
with open(os.path.join(DATA_PATH, "class_weights.json"), "w") as f:
    json.dump(class_weights, f)

In [16]:
# Validate both weighted and unweighted datasets are able to be created
train_ds_weighted = create_ds_from_df(train_df, shuffle=True, augment=True, batch_size=BATCH_SIZE, use_autotune=USE_AUTOTUNE, weighted=True)
print(f"Unweighted dataset has {len(next(iter(train_ds.take(1))))} elements")
print(f"Weighted dataset has {len(next(iter(train_ds_weighted.take(1))))} elements")

Unweighted dataset has 2 elements
Weighted dataset has 3 elements
